In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# The pandas index and Polars document_id represent the same business ids.
_HYBRID_DOCUMENT_IDS = [101, 205, 309]

# --- hybrid_scorer_citation_index ---
FIX_HYBRID_SCORER_CITATION_INDEX_CITATION_TO_LANGUAGE_CANDIDATES_PD = pd.DataFrame(
    {"document_id": _HYBRID_DOCUMENT_IDS, "score": [0.9, 0.7, 0.5]}
).set_index("document_id")
FIX_HYBRID_SCORER_CITATION_INDEX_CITATION_TO_LANGUAGE_CANDIDATES_PL = pl.DataFrame(
    {"document_id": _HYBRID_DOCUMENT_IDS, "score": [0.9, 0.7, 0.5]}
)
FIX_HYBRID_SCORER_CITATION_INDEX_HEIGHT = len(_HYBRID_DOCUMENT_IDS)

# --- hybrid_scorer_language_index ---
FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PD = pd.DataFrame(
    {"document_id": _HYBRID_DOCUMENT_IDS, "score": [0.9, 0.7, 0.5]}
).set_index("document_id")
FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PL = pl.DataFrame(
    {"document_id": _HYBRID_DOCUMENT_IDS, "score": [0.9, 0.7, 0.5]}
)

# Backward-compatible aliases for older test cells.
FIX_HYBRID_SCORER_CITATION_INDEX_CITATION_TO_LANGUAGE_CANDIDATES = FIX_HYBRID_SCORER_CITATION_INDEX_CITATION_TO_LANGUAGE_CANDIDATES_PD
FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES = FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PD

def _set_hybrid_self_before(empty=False):
    global self
    citation = FIX_HYBRID_SCORER_CITATION_INDEX_CITATION_TO_LANGUAGE_CANDIDATES_PD.copy()
    language = FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PD.copy()
    if empty:
        citation = citation.iloc[0:0]
        language = language.iloc[0:0]
    self = SimpleNamespace(
        citation_to_language_candidate_ids=[],
        citation_to_language_candidates=citation,
        language_to_citation_candidate_ids=[],
        language_to_citation_candidates=language,
    )
    return self

def _set_hybrid_self_generated(empty=False):
    global self
    citation = FIX_HYBRID_SCORER_CITATION_INDEX_CITATION_TO_LANGUAGE_CANDIDATES_PL.clone()
    language = FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PL.clone()
    if empty:
        citation = citation.head(0)
        language = language.head(0)
    self = SimpleNamespace(
        citation_to_language_candidate_ids=[],
        citation_to_language_candidates=citation,
        language_to_citation_candidate_ids=[],
        language_to_citation_candidates=language,
    )
    return self

def _ids_to_list(value):
    if isinstance(value, (pd.Index, pd.Series)):
        return value.tolist()
    if isinstance(value, pl.Series):
        return value.to_list()
    if isinstance(value, range):
        return list(value)
    if isinstance(value, (list, tuple)):
        return list(value)
    return value

_set_hybrid_self_before()
print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_hybrid_scorer_citation_index(citation_to_language_candidates, height):
    citation_to_language_candidate_ids: pd.Index = field(init=False)
    ...
    self.citation_to_language_candidate_ids = self.citation_to_language_candidates.index
    return None

def before_hybrid_scorer_language_index(language_to_citation_candidates):
    language_to_citation_candidate_ids: pd.Index = field(init=False)
    ...
    self.language_to_citation_candidate_ids = self.language_to_citation_candidates.index
    return None

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_hybrid_scorer_citation_index(citation_to_language_candidates, height):

    self.citation_to_language_candidate_ids = pl.Series(
        "index", range(self.citation_to_language_candidates.height)
    )
    return None

def gen_hybrid_scorer_language_index(language_to_citation_candidates):

    self.language_to_citation_candidate_ids = pl.Series(
        "index", range(self.language_to_citation_candidates.height)
    )
    return None

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: hybrid_scorer_language_index ===

# L1 smoke – generated
try:
    _set_hybrid_self_generated()
    _r = gen_hybrid_scorer_language_index(FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PL)
    print("✅ L1 smoke gen_hybrid_scorer_language_index: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_hybrid_scorer_language_index: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _set_hybrid_self_before()
    _rb = before_hybrid_scorer_language_index(FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PD)
    print("✅ L1 smoke before_hybrid_scorer_language_index: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_hybrid_scorer_language_index: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence — side effect on self.language_to_citation_candidate_ids
try:
    _set_hybrid_self_before()
    before_hybrid_scorer_language_index(FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PD)
    _rb = _ids_to_list(self.language_to_citation_candidate_ids)
    _set_hybrid_self_generated()
    gen_hybrid_scorer_language_index(FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PL)
    _rg = _ids_to_list(self.language_to_citation_candidate_ids)
    if _rb == _rg:
        print("✅ L2 equivalence hybrid_scorer_language_index: MATCH")
    else:
        print(f"❌ L2 equivalence hybrid_scorer_language_index: MISMATCH — before={_rb!r}, generated={_rg!r}")
except Exception as _e:
    print(f"❌ L2 equivalence hybrid_scorer_language_index: setup error — {type(_e).__name__}: {_e}")

# L3 edge - empty candidate frame; compare the mutated id collection.
try:
    _set_hybrid_self_before(empty=True)
    before_hybrid_scorer_language_index(
        FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PD.head(0)
    )
    _rb = _ids_to_list(self.language_to_citation_candidate_ids)
    _set_hybrid_self_generated(empty=True)
    gen_hybrid_scorer_language_index(
        FIX_HYBRID_SCORER_LANGUAGE_INDEX_LANGUAGE_TO_CITATION_CANDIDATES_PL.head(0)
    )
    _rg = _ids_to_list(self.language_to_citation_candidate_ids)
    if _rb == _rg:
        print("✅ L3 edge hybrid_scorer_language_index empty ids: MATCH")
    else:
        print(f"❌ L3 edge hybrid_scorer_language_index empty ids: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e: print(f"❌ L3 edge hybrid_scorer_language_index: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_hybrid_scorer_language_index: OK, type= NoneType
✅ L1 smoke before_hybrid_scorer_language_index: OK
❌ L2 equivalence hybrid_scorer_language_index: MISMATCH — before=[101, 205, 309], generated=[0, 1, 2]
✅ L3 edge hybrid_scorer_language_index empty ids: MATCH
